In [1]:
import os, sys
sys.path.append(os.path.abspath('..'))

In [ ]:
# Library imports
import json
import time
import numpy as np
import pandas as pd
import torch
import gymnasium as gym

# File imports
from env import InventoryEnv
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker

# Maximum distance between a facility and a customer
DISTANCE_NORM = 212.13

class ScaledRewardWrapper(gym.RewardWrapper):
    # Scale the reward by the maximum distance between a facility and a customer (212.13) so that the reward is in the range [0, 1] for easier learning.
    def reward(self, reward):
        return reward / DISTANCE_NORM


def _ppo_action_mask(env):
    # Only allow actions that correspond to warehouses with remaining capacity. This is used to create an action mask for the MaskablePPO algorithm.
    return np.array([capacity > 0 for capacity in env.warehouses_capacity])

In [ ]:
%load_ext tensorboard
%tensorboard --logdir='proximal_policy_optimization_training/Data_Training' --port 6006    #Change port if needed (6006,9009,9999)

In [ ]:
def train_ppo(num_warehouses, num_customers, capacity_distribution):

    SEED = 42
    np.random.seed(SEED)

    env = InventoryEnv(num_warehouses, num_customers, capacity_distribution)
    env = ActionMasker(env, _ppo_action_mask)
    env = ScaledRewardWrapper(env)

    env.reset()

    model_PPO = MaskablePPO(
        "MlpPolicy",
        env,
        tensorboard_log=f"proximal_policy_optimization_training/Data_Training/PPO/w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}/",
        verbose=0,
        seed=SEED,  # seeds SB3's own internals (policy init, action sampling, rollout buffer)
    )
    n_steps = num_customers * 50_000   

    start_time = time.perf_counter()
    model_PPO.learn(n_steps, reset_num_timesteps=True)
    training_time_minutes = (time.perf_counter() - start_time) / 60

    model_PPO.save(f'proximal_policy_optimization_training/rl_models/ppo_models/ppo_model_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.zip')
    print('Saved PPO model')

    return training_time_minutes

In [ ]:
num_warehouses_options = [2, 3, 4, 5]
num_customers_options = [50, 100, 200, 400]
capacity_distribution_options = ['uniform','uneven']

training_times = []  # one row per (num_warehouses, num_customers, capacity_distribution) family

for num_customers in num_customers_options:
    for num_warehouses in num_warehouses_options:
        for capacity_distribution in capacity_distribution_options:
            training_time_minutes = train_ppo(num_warehouses, num_customers, capacity_distribution)
            print(f"Trained model for {num_warehouses} warehouses, {num_customers} customers, {capacity_distribution} capacity")

            training_times.append({
                'num_warehouses': num_warehouses,
                'num_customers': num_customers,
                'capacity_distribution': capacity_distribution,
                'training_time': round(training_time_minutes, 2),
            })

info_dir = 'proximal_policy_optimization_training'
os.makedirs(info_dir, exist_ok=True)

training_times_df = pd.DataFrame(training_times)
training_times_df.to_csv(os.path.join(info_dir, 'proximal_policy_optimization_training_times.csv'), index=False, float_format='%.5f')